# Review the source cleaning rule
Select recordings lasting at least two seconds with at least four words.
Exports are maintained by `scripts/organize_datasets.py`, so rerunning this analysis preserves the train/test split.

In [ ]:
import torch
from torch import nn

print(torch.__version__)

In [ ]:
from pathlib import Path
import pandas as pd

root = next((parent / "data" for parent in (Path.cwd(), *Path.cwd().parents)
             if (parent / "data/karakalpak-audio-dataset-hf/sources/data.csv").is_file()), None)
if root is None:
    raise FileNotFoundError("karakalpak-audio-dataset-hf/sources/data.csv")
hf_dir = root / "karakalpak-audio-dataset-hf"
data = pd.read_csv(hf_dir / "sources/data.csv")


In [ ]:
data["word_count"] = data["sentence_latin"].fillna("").str.split().str.len()


In [ ]:
drop_mask = (data["duration"] < 2) | (data["word_count"] < 4)
print(f"Dropped {drop_mask.sum()} rows; {len(data) - drop_mask.sum()} remain")
data = data.loc[~drop_mask].reset_index(drop=True).copy()


In [ ]:
data["source_filename"] = data["filename"]
data["audio_index"] = data.index
data["filename"] = [f"audio_{i:06d}.wav" for i in data["audio_index"]]
data = data[["audio_index", "filename", "source_filename", "sentence_latin", "sentence_cyrillic", "duration", "snr_original", "snr_16k", "word_count"]]
data.head()


In [ ]:
# The organizer already exported audio and assigned both splits.
# Compare the selected source filenames; do not overwrite the exported dataset.
prepared = pd.read_csv(hf_dir / "provenance/metadata.csv")
if set(data["source_filename"]) != set(prepared["source_filename"]):
    raise ValueError("Organized HF samples differ from the cleaning rule")
prepared.groupby("split").size()


In [ ]:
train = pd.read_csv(hf_dir / "train.csv")
test = pd.read_csv(hf_dir / "test.csv")
train.head()


In [ ]:
assert set(train["audio_path"]).isdisjoint(test["audio_path"])
assert len(train) + len(test) == len(data)


In [ ]:
print(f"Selected {len(data)} recordings: {len(train)} train / {len(test)} test")


In [ ]:
data["snr_16k"].describe()

In [ ]:
data.shape